In [ ]:
import pandas as pd

df_long = pd.read_parquet("smr_hourly_library_2030_2045.parquet")

df_wide = df_long.pivot_table(
    index=["timestamp_utc", "smr_case", "year"],
    columns="unit_id",
    values="delivered_mw",
    fill_value=0.0
).reset_index()

df_wide = df_wide.rename(columns={
    "unit_1": "unit_1_mw",
    "unit_2": "unit_2_mw",
    "unit_3": "unit_3_mw"
})

for unit in ["unit_1_mw", "unit_2_mw", "unit_3_mw"]:
    if unit not in df_wide.columns:
        df_wide[unit] = 0.0

df_wide["total_fleet_mw"] = (
    df_wide["unit_1_mw"] +
    df_wide["unit_2_mw"] +
    df_wide["unit_3_mw"]
)

df_ht = df_wide.copy()
df_ht["fes_scenario"] = "Holistic Transition"

df_ee = df_wide.copy()
df_ee["fes_scenario"] = "Electric Engagement"

df_fleet = pd.concat([df_ht, df_ee], ignore_index=True)

cols_order = [
    "timestamp_utc", "fes_scenario", "smr_case", "year",
    "unit_1_mw", "unit_2_mw", "unit_3_mw", "total_fleet_mw"
]

df_fleet = df_fleet[cols_order]

df_fleet.to_parquet("smr_hourly_fleet_scenarios.parquet", index=False)

print("Owner 2 task completed! The generated fleet scenario data preview is shown below:")
print(df_fleet.head())

In [ ]:
# ouput in 2035-01-01.(checking results)

In [ ]:
import pandas as pd

df = pd.read_parquet("smr_hourly_fleet_scenarios.parquet")

print(f"Total rows: {len(df)}")

print(df["fes_scenario"].value_counts())

sample = df[
    (df["timestamp_utc"] == "2035-01-01 00:00:00") &
    (df["fes_scenario"] == "Holistic Transition")
]

print("\nSample check:")
print(sample[["timestamp_utc", "fes_scenario", "unit_1_mw", "total_fleet_mw"]])